# Syria Idleb Population Raster Masking Map (2026)

Overview

Extracts the WorldPop 2026 population raster for Idleb Governorate using polygon-based raster masking.
The national population raster is clipped to the Idleb administrative boundary and saved as a derived GeoTIFF.
The masked raster is displayed as an interactive population distribution map using logarithmic colour scaling.

WorldPop 2026のシリア全国人口Rasterを、Idleb県の行政界ポリゴンでMask処理し、対象地域を抽出します。
抽出した人口Rasterは、派生GeoTIFFとしてプロジェクト内のoutputsフォルダへ保存します。
Mask処理後のRasterは、対数カラースケールを用いたインタラクティブな人口分布地図として表示します。

Objectives

- Select Idleb Governorate as the area of interest
- Validate the administrative boundary and population raster
- Mask the national population raster using the Idleb polygon
- Apply the cell-centre rule to boundary pixels
- Preserve NoData values and spatial metadata
- Save the masked raster as a derived GeoTIFF
- Apply logarithmic colour scaling for map display
- Create an interactive Folium map with a legend and source information

- 対象地域としてIdleb県を選択する
- 行政界データと人口Rasterを検証する
- Idleb県のポリゴンで全国人口RasterをMask処理する
- 境界画素に画素中心方式を適用する
- NoData値と空間メタデータを保持する
- Mask処理したRasterを派生GeoTIFFとして保存する
- 地図表示に対数カラースケールを適用する
- 凡例と出典を備えたインタラクティブ地図を作成する

Workflow

#### English

1. Define and validate the input and output paths
2. Read the governorate boundaries and population raster metadata
3. Validate the coordinate reference systems and NoData value
4. Select and validate Idleb Governorate
5. Prepare the Idleb polygon for raster masking
6. Mask the national population raster using the cell-centre rule
7. Update and preserve the output raster metadata
8. Save and validate the derived GeoTIFF
9. Apply logarithmic colour scaling
10. Visualise the masked raster on a focused Folium map
11. Add the Idleb boundary, label, legend and source information
12. Export the interactive HTML map

#### 日本語

1. 入出力ファイルのパスを設定し、存在を確認する
2. 県境データと人口Rasterのメタデータを読み込む
3. 座標参照系とNoData値を検証する
4. Idleb県を選択し、ジオメトリを検証する
5. Raster Maskingに使用するIdlebポリゴンを準備する
6. 画素中心方式を用いて全国人口RasterをMask処理する
7. 出力Rasterのメタデータを更新・保持する
8. 派生GeoTIFFを保存し、内容を検証する
9. 対数カラースケールを適用する
10. Idleb県を中心としたFolium地図上でRasterを表示する
11. Idleb県境、ラベル、凡例および出典を追加する
12. インタラクティブHTML地図を出力する

Data

Administrative boundary data:

- `syr_admin1.geojson`
- Source: HDX OCHA, Syria subnational administrative boundaries

Population raster data:

- `worldpop_syria_2026.tif`
- Source: WorldPop 2026, Open Access Data
- Source resolution: approximately 100 m

Derived raster output:

- `outputs/worldpop_idleb_2026.tif`

Data Scope

Population values represent estimated population per source grid cell.
The mask includes source raster cells whose centres fall within the Idleb Governorate boundary.
The derived GeoTIFF retains the source raster resolution, CRS and NoData definition.

人口値は、元Rasterのグリッドセルごとの推計人口を表します。
Idleb県境内に画素中心が含まれる元RasterセルをMask処理の対象とします。
派生GeoTIFFは、元Rasterの解像度、CRSおよびNoData定義を保持します。

Technologies

- Python
- GeoPandas
- Rasterio
- NumPy
- Matplotlib
- Folium

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from pathlib import Path

import folium
import geopandas as gpd
import matplotlib.colors as mcolors
import numpy as np
import rasterio

from rasterio.mask import mask

In [ ]:
# 2
# Define and validate the input and output paths
# 入力データと出力ファイルのパスを設定し、存在を確認する

PROJECT_DIR = Path.cwd()
ROOT_DIR = PROJECT_DIR.parents[1]

VECTOR_DIR = ROOT_DIR / "02_DATA" / "VECTOR"
RASTER_DIR = ROOT_DIR / "02_DATA" / "RASTER"
OUTPUT_DIR = PROJECT_DIR / "outputs"

admin1_path = (
    VECTOR_DIR / "syr_admin1.geojson"
)

population_path = (
    RASTER_DIR / "worldpop_syria_2026.tif"
)

masked_population_path = (
    OUTPUT_DIR / "worldpop_idleb_2026.tif"
)

output_path = (
    PROJECT_DIR / "04_syria_idleb_mask.html"
)

input_paths = {
    "Governorate boundaries": admin1_path,
    "Population raster": population_path,
}

missing_input_paths = [
    str(path)
    for path in input_paths.values()
    if not path.exists()
]

if missing_input_paths:
    raise FileNotFoundError(
        "The following input files were not found: "
        f"{missing_input_paths}"
    )

print(
    "Input path validation: passed"
)

In [ ]:
# 3
# Read and validate the spatial datasets
# 行政界データと人口Rasterの構造を読み込み、検証する

admin1 = gpd.read_file(
    admin1_path
)

required_admin1_columns = {
    "adm1_name",
    "adm1_pcode",
    "geometry",
}

missing_admin1_columns = (
    required_admin1_columns
    - set(admin1.columns)
)

if missing_admin1_columns:
    raise ValueError(
        "The governorate boundary dataset is missing "
        "required columns: "
        f"{sorted(missing_admin1_columns)}"
    )

if admin1.empty:
    raise ValueError(
        "The governorate boundary dataset contains no features."
    )

with rasterio.open(population_path) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds
    population_nodata = src.nodata

if admin1.crs is None:
    raise ValueError(
        "The governorate boundary dataset has no defined CRS."
    )

if raster_crs is None:
    raise ValueError(
        "The population raster has no defined CRS."
    )

if admin1.crs != raster_crs:
    raise ValueError(
        "The governorate boundaries and population raster "
        "must use the same CRS. "
        f"Vector CRS: {admin1.crs}; "
        f"Raster CRS: {raster_crs}"
    )

if population_nodata is None:
    raise ValueError(
        "The population raster has no defined NoData value."
    )

print(
    "Spatial dataset validation: passed"
)

print(
    f"Governorate boundary CRS: {admin1.crs}"
)

print(
    f"Population raster CRS: {raster_crs}"
)

print(
    f"Population raster NoData: {population_nodata}"
)

print(
    f"Population raster bounds: {raster_bounds}"
)

In [ ]:
# 4
# Select and validate Idleb Governorate
# Idleb県を選択し、ジオメトリを検証する

idleb = admin1[
    admin1["adm1_name"] == "Idleb"
].copy()

if len(idleb) != 1:
    raise ValueError(
        "Exactly one Idleb Governorate feature was expected, "
        f"but {len(idleb)} features were selected."
    )

if idleb.geometry.isna().any():
    raise ValueError(
        "The Idleb Governorate geometry is missing."
    )

if idleb.geometry.is_empty.any():
    raise ValueError(
        "The Idleb Governorate geometry is empty."
    )

if not idleb.geometry.is_valid.all():
    raise ValueError(
        "The Idleb Governorate geometry is invalid."
    )

print(
    idleb[
        [
            "adm1_name",
            "adm1_pcode",
        ]
    ]
)

In [ ]:
# 5
# Prepare the Idleb masking geometry
# Idleb県のジオメトリをRaster Masking用に変換する

idleb_masking_geometry = [
    geometry.__geo_interface__
    for geometry in idleb.geometry
]

In [ ]:
# 6
# Mask the national population raster
# 画素中心方式を用いてIdleb県の人口Rasterを抽出する

with rasterio.open(population_path) as src:

    out_image, out_transform = mask(
        src,
        idleb_masking_geometry,
        crop=True,
        all_touched=False,
        nodata=population_nodata,
        filled=True,
    )

    out_meta = src.meta.copy()

In [ ]:
# 7
# Update the masked raster metadata
# Mask処理後のRasterメタデータを更新する

if out_image.ndim != 3:
    raise ValueError(
        "The masked raster must contain band, row "
        "and column dimensions."
    )

if out_image.shape[1] == 0 or out_image.shape[2] == 0:
    raise ValueError(
        "The masked raster has no rows or columns."
    )

out_meta.update({
    "driver": "GTiff",
    "count": out_image.shape[0],
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform,
    "crs": raster_crs,
    "nodata": population_nodata,
    "compress": "lzw",
})

print(
    f"Masked raster dimensions: "
    f"{out_image.shape[2]:,} × {out_image.shape[1]:,}"
)

In [ ]:
# 8
# Save the masked raster as a derived GeoTIFF
# Mask処理したRasterを派生GeoTIFFとして保存する

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

with rasterio.open(
    masked_population_path,
    "w",
    **out_meta,
) as dest:
    dest.write(
        out_image
    )

if not masked_population_path.exists():
    raise FileNotFoundError(
        "The derived Idleb population raster was not saved."
    )

print(
    f"Masked raster saved to: {masked_population_path}"
)

In [ ]:
# 9
# Create a light basemap focused on Idleb Governorate
# Idleb県を中心とした地名表記のない白色ベースマップを作成する

m = folium.Map(
    location=[35.9, 36.6],
    zoom_start=8,
    tiles=None,
)

folium.TileLayer(
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        '&copy; <a href="https://carto.com/attributions">CARTO</a>'
    ),
    name="CARTO Light — No Labels",
    overlay=False,
    control=True,
).add_to(
    m
)

min_x, min_y, max_x, max_y = (
    idleb.total_bounds
)

idleb_view_bounds = [
    [min_y, min_x],
    [max_y, max_x],
]

m.fit_bounds(
    idleb_view_bounds,
    padding=(40, 40),
    max_zoom=9,
)

In [ ]:
# 10
# Read and validate the derived population raster
# 保存したIdleb人口Rasterを読み込み、空間情報を検証する

with rasterio.open(masked_population_path) as src:

    idleb_masked_array = src.read(
        1,
        masked=True,
    ).astype(
        "float32"
    )

    masked_raster_crs = src.crs
    masked_raster_nodata = src.nodata
    masked_raster_count = src.count

    idleb_bounds = [
        [
            src.bounds.bottom,
            src.bounds.left,
        ],
        [
            src.bounds.top,
            src.bounds.right,
        ],
    ]

if masked_raster_crs != raster_crs:
    raise ValueError(
        "The derived Idleb raster CRS does not match "
        "the source population raster CRS."
    )

if masked_raster_nodata != population_nodata:
    raise ValueError(
        "The derived Idleb raster NoData value does not match "
        "the source population raster NoData value."
    )

if masked_raster_count != 1:
    raise ValueError(
        "The derived Idleb population raster must contain "
        f"one band, but {masked_raster_count} bands were found."
    )

idleb_array = idleb_masked_array.filled(
    np.nan
)

print(
    "Derived raster validation: passed"
)

print(
    f"Derived raster CRS: {masked_raster_crs}"
)

print(
    f"Derived raster NoData: {masked_raster_nodata}"
)

print(
    f"Derived raster shape: {idleb_array.shape}"
)

print(
    f"Derived raster bounds: {idleb_bounds}"
)

In [ ]:
# 11
# Validate and summarise the masked population values
# Mask処理した人口値を検証し、基本統計量を集計する

valid_idleb_values = idleb_array[
    np.isfinite(
        idleb_array
    )
]

if valid_idleb_values.size == 0:
    raise ValueError(
        "The derived Idleb raster contains no valid values."
    )

negative_value_count = int(
    np.count_nonzero(
        valid_idleb_values < 0
    )
)

if negative_value_count > 0:
    raise ValueError(
        "The derived Idleb raster contains "
        f"{negative_value_count:,} negative population values."
    )

idleb_min = float(
    valid_idleb_values.min()
)

idleb_max = float(
    valid_idleb_values.max()
)

idleb_mean = float(
    valid_idleb_values.mean()
)

idleb_population_sum = float(
    valid_idleb_values.sum()
)

print(
    "Population value validation: passed"
)

print(
    f"Valid population cells: {valid_idleb_values.size:,}"
)

print(
    f"Minimum population value: {idleb_min:,.2f}"
)

print(
    f"Maximum population value: {idleb_max:,.2f}"
)

print(
    f"Mean population value: {idleb_mean:,.2f}"
)

print(
    "Estimated Idleb population total: "
    f"{idleb_population_sum:,.0f}"
)

In [ ]:
# 12
# Create a NoData-aware population colour map
# NoDataを透明にする人口カラーマップを作成する

green_cmap = (
    mcolors.LinearSegmentedColormap.from_list(
        "Idleb_Population_Green",
        [
            (0.00, "#ffffff00"),
            (0.25, "#d9f0d3ff"),
            (0.50, "#1d5c37ff"),
            (0.75, "#00441bff"),
            (1.00, "#01240fff"),
        ],
    )
)

green_cmap.set_bad(
    (0, 0, 0, 0)
)

In [ ]:
# 13
# Create logarithmic population normalisation
# 人口値を対数変換し、カラースケールへ割り当てる

if not np.isfinite(idleb_max) or idleb_max <= 0:
    raise ValueError(
        "The population raster maximum must be "
        "a finite value greater than zero."
    )

log_population_max = np.log1p(
    idleb_max
)

log_norm = mcolors.Normalize(
    vmin=0,
    vmax=log_population_max,
    clip=True,
)


def population_log_colormap(value):
    """
    Convert a population value to a logarithmically
    scaled RGBA colour.

    人口値を対数スケールのRGBAカラーへ変換する。
    """

    if (
        value is None
        or not np.isfinite(value)
        or value <= 0
    ):
        return (0, 0, 0, 0)

    log_value = np.log1p(
        value
    )

    return green_cmap(
        log_norm(
            log_value
        )
    )

In [ ]:
# 14
# Add the masked population raster layer
# Mask処理したIdleb人口Rasterを地図へ追加する

folium.raster_layers.ImageOverlay(
    image=idleb_array,
    bounds=idleb_bounds,
    colormap=population_log_colormap,
    opacity=0.85,
    name="Idleb Population Distribution",
    overlay=True,
    control=True,
    show=True,
).add_to(
    m
)

In [ ]:
# 15
# Add the Idleb Governorate label
# Idleb県名を独立したラベルレイヤーとして追加する

idleb_label_point = (
    idleb.geometry.iloc[0]
    .representative_point()
)

idleb_label_group = folium.FeatureGroup(
    name="Idleb Label",
    overlay=True,
    control=True,
    show=True,
)

folium.Marker(
    location=[
        idleb_label_point.y,
        idleb_label_point.x,
    ],
    icon=folium.DivIcon(
        html="""
        <div style="
            font-size: 20pt;
            color: #333333;
            font-weight: bold;
            white-space: nowrap;
            text-align: center;
            width: 120px;
            margin-left: -60px;
            text-shadow:
                -1px -1px 0 white,
                1px -1px 0 white,
                -1px 1px 0 white,
                1px 1px 0 white;
        ">
            Idleb
        </div>
        """
    ),
).add_to(
    idleb_label_group
)

idleb_label_group.add_to(
    m
)

In [ ]:
# 16
# Prepare the Idleb boundary layer
# 県境表示に必要な属性とジオメトリだけを抽出する

idleb_map = idleb[
    [
        "adm1_name",
        "adm1_pcode",
        "geometry",
    ]
].copy()

In [ ]:
# 17
# Add the Idleb Governorate boundary
# Idleb県境レイヤーを地図へ追加する

folium.GeoJson(
    idleb_map,
    name="Idleb Governorate Boundary",
    style_function=lambda feature: {
        "color": "#333333",
        "weight": 2,
        "fillOpacity": 0,
    },
    highlight_function=lambda feature: {
        "color": "#1d5c37",
        "weight": 3,
        "fillOpacity": 0.04,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(
    m
)

In [ ]:
# 18
# Add the map information panel
# 地図の概要、Mask処理、集計結果および出典を追加する

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 410px;
    min-height: 205px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Idleb Governorate
    </b>
    <br>

    <span style="
        color: #1d5c37;
        font-weight: bold;
    ">
        Population Raster Masking (2026)
    </span>

    <small style="
        display: block;
        margin-top: 7px;
        line-height: 1.35;
        color: #333333;
    ">
        The national WorldPop 2026 population raster
        was masked using the Idleb Governorate boundary.
        Source grid cells whose centres fall within the
        boundary are included. Colours represent estimated
        population per source grid cell using logarithmic
        scaling. The derived raster retains the source
        resolution of approximately 100 m.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        Estimated population:
        <b>{idleb_population_sum:,.0f}</b>
        <br>

        Population source:
        <a
            href="https://hub.worldpop.org/geodata/summary?id=75632"
            target="_blank"
            style="
                color: #1d5c37;
                text-decoration: none;
                font-weight: bold;
            "
        >
            WorldPop 2026
        </a>
        <br>

        Boundary source: HDX OCHA
        <br>

        Method: Polygon Raster Masking /
        Cell-Centre Rule / Logarithmic Display
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        information_panel_html
    )
)

In [ ]:
# 19
# Add the logarithmic population colour legend
# 人口値と色の濃さを対応させた対数スケール凡例を追加する

legend_positions = [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
]

legend_values = [
    float(
        np.expm1(
            position * log_population_max
        )
    )
    for position in legend_positions
]

legend_labels = [
    f"{value:,.0f}"
    for value in legend_values
]

legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px;
    right: 40px;
    width: 310px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    font-size: 12px;
    z-index: 9999;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.2);
">
    <b>
        Estimated population per source grid cell
    </b>

    <div style="
        height: 16px;
        margin-top: 10px;
        border: 1px solid #777777;
        background: linear-gradient(
            to right,
            rgba(255, 255, 255, 0),
            #d9f0d3 25%,
            #1d5c37 50%,
            #00441b 75%,
            #01240f 100%
        );
    "></div>

    <div style="
        display: flex;
        justify-content: space-between;
        margin-top: 4px;
        font-size: 10px;
        font-weight: bold;
    ">
        <span>{legend_labels[0]}</span>
        <span>{legend_labels[1]}</span>
        <span>{legend_labels[2]}</span>
        <span>{legend_labels[3]}</span>
        <span>{legend_labels[4]}</span>
    </div>

    <div style="
        margin-top: 9px;
        padding-top: 6px;
        color: #555555;
        border-top: 1px solid #aaaaaa;
        line-height: 1.35;
    ">
        Darker colours indicate higher estimated
        population values.
        <br>
        Colours use logarithmic scaling.
        <br>
        Source raster resolution: approximately 100 m
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        legend_html
    )
)

In [ ]:
# 20
# Add the layer control
# 地図レイヤーの表示と非表示を切り替える機能を追加する

folium.LayerControl(
    collapsed=False,
).add_to(
    m
)

In [ ]:
# 21
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(
    output_path
)

print(
    f"Map saved to: {output_path}"
)

m